# CyberGuard — Agentic AI Cybersecurity Research System

In this notebook, we build **CyberGuard**, an agentic AI system for cybersecurity research.

The system combines multiple agents and tools to investigate a cybersecurity question, retrieve relevant internal security policies, analyze evidence, verify the findings, and generate a structured final report.

The workflow includes:

* **Planner** — breaks the user's question into focused research tasks.
* **Researcher** — gathers information from external web sources.
* **Policy RAG** — retrieves relevant internal cybersecurity policies.
* **Analyst** — analyzes and combines the collected evidence.
* **Fact Checker** — verifies the findings and determines whether additional research is needed.
* **Writer** — produces the final cybersecurity report.
* **Loop Detection** — helps prevent repeated tool calls and stagnation.
* **Observability** — tracks execution, timing, and model usage.

The project was developed using **LangChain, LangGraph, OpenRouter, Tavily, and a local vector store**.



In [1]:
# On your own machine (not Colab) run: uv sync — and skip this cell
!pip install -q "langchain>=1.4" "langgraph>=1.2.11" "langchain-openai>=1.2" beautifulsoup4 requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 1.3 MB/s eta 0:00:00


## 1. Setup

This section configures the OpenRouter API connection and initializes the language model used by CyberGuard.


In [2]:
import getpass
import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

try:
    # Running in Google Colab? Read the key from Colab Secrets.
    from google.colab import userdata
    try:
        # userdata.get raises SecretNotFoundError if the secret is not set;
        # fall through to the getpass prompt below instead of erroring.
        api_key = userdata.get('OPENROUTER_API_KEY') or ''
    except Exception:
        api_key = ''
except ImportError:
    # Running locally? Read the key from a .env file.
    load_dotenv()
    api_key = os.environ.get('OPENROUTER_API_KEY', '')

if not api_key:
    # Last resort: ask for the key in this cell.
    api_key = getpass.getpass('Paste your OPENROUTER_API_KEY: ')
if not api_key:
    raise ValueError('Set OPENROUTER_API_KEY (Colab Secrets, .env, or paste it when asked)')

MODEL_NAME = 'apodex/apodex-1.1-mini:free'
llm = ChatOpenAI(model=MODEL_NAME, base_url='https://openrouter.ai/api/v1', api_key=api_key)

## 2. Observability helpers (given, run as-is)

Every agent run below prints a trace tree: one line per run, with the
token count and the real cost on it. Later you can swap this stub for real
Langfuse by changing one import — the function names match Langfuse's API.

In [3]:
# A small stand-in for Langfuse's @observe decorator.
# Swap to real Langfuse later by changing only the import, e.g.:
#     from langfuse import observe
import asyncio
import functools
import time
import uuid
from contextvars import ContextVar
from dataclasses import dataclass, field
from typing import Any, List, Optional


@dataclass
class Span:
    id: str
    name: str
    start_time: float
    level: int
    type: str = 'span'
    parent: Optional['Span'] = None
    children: List['Span'] = field(default_factory=list)
    end_time: Optional[float] = None
    input: Any = None
    output: Any = None
    metadata: dict = field(default_factory=dict)
    usage: dict = field(default_factory=dict)
    model: Optional[str] = None


_current_span: ContextVar[Optional[Span]] = ContextVar('current_span', default=None)


def print_tree(span: 'Span'):
    """Print one span and all of its children, indented by depth."""
    duration = (span.end_time - span.start_time) * 1000
    indent = '  ' * span.level
    prefix, suffix = ('=== TRACE: ', ' ===') if span.level == 0 else ('|-- ', '')
    meta_parts = []
    if span.model:
        meta_parts.append(f'model={span.model}')
    if span.usage.get('total_tokens'):
        meta_parts.append(f"tokens={span.usage['total_tokens']}")
    if 'cost_usd' in span.metadata:
        meta_parts.append(f"${span.metadata['cost_usd']:.4f}")
    meta_str = f" [{', '.join(meta_parts)}]" if meta_parts else ''
    type_str = f" [{span.type}]" if span.type != 'span' else ''
    print(f'{indent}{prefix}{span.name}{type_str}{suffix} ({duration:.2f}ms){meta_str}')
    for child in span.children:
        print_tree(child)


def _make_span(span_name: str, span_type: str, args, kwargs) -> 'Span':
    parent = _current_span.get()
    level = parent.level + 1 if parent else 0
    span = Span(id=str(uuid.uuid4())[:8], name=span_name, type=span_type,
                start_time=time.time(), level=level, parent=parent,
                input={'args': args, 'kwargs': kwargs})
    if parent:
        parent.children.append(span)
    return span


def _finish_span(span: 'Span'):
    span.end_time = time.time()
    if span.level == 0:
        print('\n' + '-' * 60)
        print_tree(span)
        print('-' * 60 + '\n')


def observe(name=None, as_type=None):
    """Decorator that records a span around a sync or async function."""
    def decorator(func):
        span_name = name if isinstance(name, str) else func.__name__
        span_type = as_type or 'span'

        if asyncio.iscoroutinefunction(func):
            @functools.wraps(func)
            async def wrapper(*args, **kwargs):
                span = _make_span(span_name, span_type, args, kwargs)
                token = _current_span.set(span)
                try:
                    result = await func(*args, **kwargs)
                    if span.output is None:
                        span.output = result
                    return result
                except Exception as e:
                    span.output = f'Error: {e}'
                    raise
                finally:
                    _finish_span(span)
                    _current_span.reset(token)
        else:
            @functools.wraps(func)
            def wrapper(*args, **kwargs):
                span = _make_span(span_name, span_type, args, kwargs)
                token = _current_span.set(span)
                try:
                    result = func(*args, **kwargs)
                    if span.output is None:
                        span.output = result
                    return result
                except Exception as e:
                    span.output = f'Error: {e}'
                    raise
                finally:
                    _finish_span(span)
                    _current_span.reset(token)

        return wrapper

    # Support both @observe and @observe(name='foo', as_type='bar')
    if callable(name):
        func, name = name, None
        return decorator(func)
    return decorator


class LangfuseContext:
    """Attach usage/model/cost data to the span currently being recorded."""

    def update_current_observation(self, **kwargs):
        span = _current_span.get()
        if not span:
            return
        if isinstance(kwargs.get('usage'), dict):
            span.usage.update(kwargs['usage'])
        if 'model' in kwargs:
            span.model = kwargs['model']
        if isinstance(kwargs.get('metadata'), dict):
            span.metadata.update(kwargs['metadata'])


langfuse_context = LangfuseContext()

In [4]:
# LoopDetector: spots when an agent repeats itself.
# check_tool_call flags repeated tool calls; check_output_stagnation flags
# stage outputs that keep coming back nearly identical.
from dataclasses import dataclass


@dataclass
class LoopDetectionResult:
    is_looping: bool
    strategy: str  # 'exact', 'fuzzy', 'stagnation', or 'none'
    message: str
    confidence: float


class LoopDetector:
    """Detects agent loops using exact match, fuzzy match, and stagnation."""

    def __init__(self, exact_threshold: int = 2, fuzzy_threshold: float = 0.8,
                 stagnation_window: int = 3):
        self.exact_threshold = exact_threshold
        self.fuzzy_threshold = fuzzy_threshold
        self.stagnation_window = stagnation_window
        self.tool_history: list[tuple[str, str]] = []  # (tool_name, args_str)
        self.output_history: list[str] = []

    def _jaccard_similarity(self, s1: str, s2: str) -> float:
        """Word-level overlap between two strings: shared words / all words."""
        tokens1 = set(s1.lower().split())
        tokens2 = set(s2.lower().split())
        if not tokens1 and not tokens2:
            return 1.0
        if not tokens1 or not tokens2:
            return 0.0
        return len(tokens1 & tokens2) / len(tokens1 | tokens2)

    def check_tool_call(self, tool_name: str, tool_input: str) -> LoopDetectionResult:
        """Call this BEFORE running a tool. Flags identical or near-identical repeats."""
        current = (tool_name, tool_input.strip())
        exact_count = sum(1 for past_tool, past_input in self.tool_history
                          if (past_tool, past_input.strip()) == current)
        if exact_count >= self.exact_threshold:
            self.tool_history.append(current)
            return LoopDetectionResult(
                is_looping=True, strategy='exact', confidence=1.0,
                message=(f"Exact loop detected: '{tool_name}' called {exact_count + 1} "
                         f'times with identical arguments. Change your approach.'))

        recent_history = self.tool_history[-5:]
        fuzzy_matches = sum(1 for past_tool, past_input in recent_history
                            if past_tool == tool_name
                            and self._jaccard_similarity(tool_input, past_input) >= self.fuzzy_threshold)
        if fuzzy_matches >= self.exact_threshold:
            self.tool_history.append(current)
            return LoopDetectionResult(
                is_looping=True, strategy='fuzzy', confidence=0.85,
                message=(f"Fuzzy loop detected: '{tool_name}' called with very similar "
                         f'arguments {fuzzy_matches + 1} times. Try a different approach.'))

        self.tool_history.append(current)
        return LoopDetectionResult(is_looping=False, strategy='none', message='', confidence=0.0)

    def check_output_stagnation(self, output: str) -> LoopDetectionResult:
        """Flags when the last few outputs are all nearly the same text."""
        self.output_history.append(output)
        if len(self.output_history) < self.stagnation_window:
            return LoopDetectionResult(is_looping=False, strategy='none', message='', confidence=0.0)

        recent = self.output_history[-self.stagnation_window:]
        similarities = [self._jaccard_similarity(recent[i], recent[j])
                        for i in range(len(recent)) for j in range(i + 1, len(recent))]
        avg_similarity = sum(similarities) / len(similarities) if similarities else 0
        if avg_similarity >= self.fuzzy_threshold:
            return LoopDetectionResult(
                is_looping=True, strategy='stagnation', confidence=avg_similarity,
                message=(f'Output stagnation detected: last {self.stagnation_window} '
                         f'outputs are {avg_similarity:.0%} similar. The agent is not '
                         f'making progress. Try a different approach entirely.'))
        return LoopDetectionResult(is_looping=False, strategy='none', message='', confidence=0.0)

    def reset(self):
        self.tool_history.clear()
        self.output_history.clear()

## 3. Research tools (given)

The `@tool` decorator turns a plain Python function into a tool the agent
can call. The function's name, docstring, and type hints become the schema
the model sees, and the returned string becomes the tool's result. This
cell is complete — run it.

In [6]:
import logging
import socket
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup

logger = logging.getLogger(__name__)


def validate_url(url: str) -> bool:
    """
    Validate URL to prevent SSRF.
    Blocks localhost, private IP ranges, and non-http/https schemes.
    """
    try:
        parsed = urlparse(url)

        if parsed.scheme not in ["http", "https"]:
            return False

        hostname = parsed.hostname

        if not hostname:
            return False

        try:
            ip_address = socket.gethostbyname(hostname)
        except socket.gaierror:
            return False

        parts = ip_address.split(".")

        if parts[0] == "10":
            return False

        if parts[0] == "192" and parts[1] == "168":
            return False

        if parts[0] == "172" and 16 <= int(parts[1]) <= 31:
            return False

        if parts[0] == "127":
            return False

        if ip_address == "0.0.0.0":
            return False

        return True

    except Exception:
        return False


# Get Tavily API key from Colab Secrets
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")


@tool("search_web")
def search_web(query: str, max_results: int = 5) -> str:
    """
    Search the web using Tavily.
    Returns titles, links, and snippets.
    """

    if not TAVILY_API_KEY:
        return "Search failed: TAVILY_API_KEY not found."

    url = "https://api.tavily.com/search"

    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "search_depth": "basic",
        "max_results": max_results,
        "include_answer": False
    }

    try:
        response = requests.post(
            url,
            json=payload,
            timeout=20
        )

        response.raise_for_status()

        data = response.json()

    except Exception as e:
        logger.error(f"Tavily search failed: {e}")
        return f"Search failed: {e}"

    results = data.get("results", [])

    if not results:
        return f"No results found for '{query}'."

    lines = []

    for result in results:

        title = result.get("title", "No title")
        link = result.get("url", "")
        snippet = result.get("content", "No description")

        if not link or not validate_url(link):
            continue

        lines.append(
            f"{len(lines) + 1}. {title}\n"
            f"   Link: {link}\n"
            f"   Snippet: {snippet}"
        )

    if not lines:
        return f"No valid results found for '{query}'."

    logger.info(
        f"Tavily returned {len(lines)} results for '{query}'"
    )

    return "\n\n".join(lines)


@tool("read_webpage")
def read_webpage(url: str) -> str:
    """
    Read the content of a webpage.
    Returns the text content.
    """

    if not validate_url(url):
        return "Error: Invalid or restricted URL. Access to local/private networks is blocked."

    try:

        if "example.com" in url:
            return f"Simulated content for {url}."

        logger.info(f"Reading webpage: {url}")

        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for script in soup(["script", "style"]):
            script.decompose()

        text = soup.get_text(separator="\n")

        lines = (
            line.strip()
            for line in text.splitlines()
        )

        chunks = (
            phrase.strip()
            for line in lines
            for phrase in line.split("  ")
        )

        text = "\n".join(
            chunk
            for chunk in chunks
            if chunk
        )

        content = text[:10000]

        logger.info(
            f"Read {len(content)} chars from {url}"
        )

        return content

    except Exception as e:
        return f"Error reading {url}: {e}"


RESEARCH_TOOLS = [
    search_web,
    read_webpage
]

print("Research tools updated successfully!")

Research tools updated successfully!


test for search

In [7]:
test_search = search_web.invoke({
    "query": "CISA phishing protection best practices"
})

print(test_search)

1. Phishing Guidance: Stopping the Attack Cycle at Phase One
   Link: https://www.cisa.gov/sites/default/files/2025-03/Phishing%20Guidance%20-%20Stopping%20the%20Attack%20Cycle%20at%20Phase%20One%20508.pdf
   Snippet: SMALL- AND MEDIUM-SIZED BUSINESSES (SMBs) OR ORGANIZATIONS CISA, NSA, FBI, and MS-ISAC recommend that small- and medium-sized organizations with limited resources prioritize the following best practices to protect network resources from prevalent phishing threats: • User phishing awareness training: Implement a standard anti-phishing training program and require employees to review phishing training material annually. Additionally, conclude the CISA | NSA | FBI | MS-ISAC TLP:CLEAR 8 PHISHING [...] Additionally, CISA, NSA, FBI, and MS-ISAC recommend SMBs implement the technical solutions below to prevent phishing related compromises: • Implement strong password policies to authenticate users. These passwords must adhere to a password strength policy which requires minimum 

## 4. The shared runner (given)

`run_agent` is the one function that calls any agent you build. It invokes
a compiled agent, pulls the final answer off the message list, adds up the
token usage, and records usage and the real cost from OpenRouter on the trace. It returns:

```python
{'answer': <final text>, 'metadata': {'total_messages': n, 'total_tokens': n}}
```

You will call `run_agent` from the pipeline nodes you write in TODO #3.

In [8]:
@observe(name='agent_run', as_type='agent')
async def run_agent(agent, query: str, max_steps: int = 10) -> dict:
    result = await agent.ainvoke(
        {'messages': [('user', query)]},
        config={'recursion_limit': 2 * max_steps + 1},
    )
    messages = result['messages']
    answer = messages[-1].content
    total_tokens = sum((m.usage_metadata or {}).get('total_tokens', 0) for m in messages if getattr(m, 'usage_metadata', None))
    langfuse_context.update_current_observation(usage={'total_tokens': total_tokens}, model=MODEL_NAME)
    # Real cost in USD: OpenRouter returns it in each response's usage object.
    total_cost = sum((m.response_metadata or {}).get('token_usage', {}).get('cost') or 0.0 for m in messages)
    langfuse_context.update_current_observation(metadata={'cost_usd': round(total_cost, 6)})
    return {'answer': answer, 'metadata': {'total_messages': len(messages), 'total_tokens': total_tokens}}

## 5. TODO #1 — prompts

Each agent's behavior comes from its system prompt:

- **Researcher**: gathers raw facts. It should call the web tools, collect
  relevant information, and report its findings with the links it used.
- **Analyst**: reads the research notes. It should find patterns, compare
  the points, and write a short structured analysis. It has no tools.
- **Writer**: reads the analysis. It should turn it into a clear final
  report for a beginner reader, with headings and a conclusion.

In [9]:
# TODO #1: CyberGuard researcher prompt
RESEARCHER_PROMPT = """
You are the CyberGuard Researcher, a cybersecurity research agent.

Your job is to investigate the user's cybersecurity question using the
available search_web and read_webpage tools.

IMPORTANT RULES:
- Use a maximum of 3 web searches for each research task.
- Prefer broad, focused searches instead of many similar searches.
- Do not repeatedly search the same topic.
- If a search returns no results, try one simpler query and then move on.
- Use read_webpage only for the most important sources.
- Prefer reliable sources such as CISA, NIST, government agencies,
  security organizations, official vendor documentation, and reputable
  security research.
- Do not make up facts or sources.
- Clearly separate verified facts from uncertain information.
- Include the source title and URL when available.
- Return concise research notes.
"""

# TODO #1: CyberGuard analyst prompt
ANALYST_PROMPT = """
You are the CyberGuard Analyst.

Your job is to analyze the research notes provided by the researcher.

Instructions:
- Do not use tools.
- Identify the most important facts and findings.
- Compare information when multiple sources disagree.
- Highlight important cybersecurity risks, causes, impacts, and recommendations.
- Do not invent information that is not supported by the research notes.
- Produce a short, clear, structured analysis.
"""

# TODO #1: CyberGuard writer prompt
WRITER_PROMPT = """
You are the CyberGuard Report Writer.

Your job is to turn the research analysis into a clear final report for a beginner
reader.

Instructions:
- Use simple and understandable language.
- Organize the report with clear headings and bullet points when useful.
- Explain important cybersecurity terms briefly.
- Include the main findings, risks, and practical recommendations.
- Keep the report concise but useful.
- Preserve the source links provided in the research.
- Do not invent facts or sources.
"""

## 6. TODO #2 — build the three agents

`create_agent` gives you the complete agent loop: a model node, a tools
node, and the routing between them. You do NOT write the loop yourself —
the model decides when to call a tool and when to answer.
`ToolCallLimitMiddleware` is the step budget: it stops the agent after a
fixed number of tool calls.

Here are the exact shapes to build (the analyst and writer have no tools,
so they need no middleware):

```python
researcher = create_agent(
    model=llm,
    tools=RESEARCH_TOOLS,
    system_prompt=RESEARCHER_PROMPT,
    middleware=[ToolCallLimitMiddleware(thread_limit=10, exit_behavior='end')],
)
analyst = create_agent(model=llm, tools=[], system_prompt=ANALYST_PROMPT)
writer = create_agent(model=llm, tools=[], system_prompt=WRITER_PROMPT)
```

انشاء الوكلاء

In [10]:
researcher = create_agent(
    model=llm,
    tools=RESEARCH_TOOLS,
    system_prompt=RESEARCHER_PROMPT,
    middleware=[ToolCallLimitMiddleware(thread_limit=10, exit_behavior='end')],
)

analyst = create_agent(
    model=llm,
    tools=[],
    system_prompt=ANALYST_PROMPT
)

writer = create_agent(
    model=llm,
    tools=[],
    system_prompt=WRITER_PROMPT
)

## 7. TODO #3 — build the pipeline with the Graph API

`create_agent` builds ONE agent. To chain several agents into a workflow,
use LangGraph's Graph API: `StateGraph`. You define a shared state (a
`TypedDict`), one node function per stage, and the edges that say what
runs after what. The pipeline is: researcher → analyst → writer.

Here is the recipe:

```python
class PipelineState(TypedDict):
    query: str    # the original question
    report: str   # the text passed from stage to stage

async def researcher_node(state: PipelineState) -> dict:
    result = await run_agent(researcher, state['query'], 10)
    return {'report': result['answer']}

async def analyst_node(state: PipelineState) -> dict:
    result = await run_agent(analyst, state['report'], 10)
    return {'report': result['answer']}

async def writer_node(state: PipelineState) -> dict:
    result = await run_agent(writer, state['report'], 10)
    return {'report': result['answer']}

graph = StateGraph(PipelineState)
graph.add_node('researcher', researcher_node)
graph.add_node('analyst', analyst_node)
graph.add_node('writer', writer_node)
graph.add_edge(START, 'researcher')
graph.add_edge('researcher', 'analyst')
graph.add_edge('analyst', 'writer')
graph.add_edge('writer', END)
pipeline = graph.compile()

result = await pipeline.ainvoke({'query': query, 'report': ''})
```

Each node returns a dict of state updates; LangGraph merges them into the
shared state before the next node runs.

**The sequential pipeline above is a starting point — it cannot pass on
its own.** A straight line caps your architecture grade below the
passing threshold (see EVALUATION.md). To pass, change the graph: run
stages in parallel, send the writer's draft back to the researcher when
the fact-checker is not satisfied, or add a planner node that splits the
query first.

## 8. Check your run

Every run above printed a trace tree: one span per agent run, with the
token count and the real cost on it. This cell prints the pipeline metadata.

In [11]:
if 'pipeline_result' not in globals():
    print('Finish TODO #2 and TODO #3 first, then run the pipeline cell.')
else:
    print('Pipeline metadata:', pipeline_result['metadata'])
    print('The trace trees above show the tokens and real cost for each stage.')

Finish TODO #2 and TODO #3 first, then run the pipeline cell.


In [12]:
PLANNER_PROMPT = """
You are the CyberGuard Planner, a cybersecurity research planning agent.

Your job is to break the user's cybersecurity question into 2-4 clear research tasks.

For each task:
- State exactly what information needs to be investigated.
- Identify whether it requires current external web research.
- Keep the tasks focused and non-overlapping.

Return only a numbered list of research tasks.
Do not answer the user's question.
"""

planner = create_agent(
    model=llm,
    tools=[],
    system_prompt=PLANNER_PROMPT
)

print("Planner created successfully.")

Planner created successfully.


# **test for planner**

In [56]:
async def test_planner(query: str):
    result = await run_agent(planner, query, 5)
    return result["answer"]

planner_result = await test_planner(
    "How can an organization protect itself from phishing attacks?"
)

print(planner_result)


------------------------------------------------------------
=== TRACE: agent_run [agent] === (1916.45ms)
------------------------------------------------------------



OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791504000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3KE0wNvJLBuuR1FkYfZ8lzr3dZw'}

In [14]:
FACT_CHECKER_PROMPT = """
You are the CyberGuard Fact Checker.

Your job is to evaluate cybersecurity research findings before a final report is written.

Check:
- Are the important claims supported by the research?
- Are there contradictions between sources?
- Is important information missing?
- Are the sources relevant and trustworthy?

At the end, return exactly one decision:
PASS
or
RESEARCH_AGAIN

Then briefly explain the decision.

Do not invent facts or sources.
"""

fact_checker = create_agent(
    model=llm,
    tools=[],
    system_prompt=FACT_CHECKER_PROMPT
)

print("Fact Checker created successfully.")

Fact Checker created successfully.


In [69]:
test_query = """
How can an organization protect itself from phishing attacks?
"""

final_result = await cyberguard_graph_final.ainvoke(
    {
        "query": test_query,
        "plan": "",
        "research": "",
        "analysis": "",
        "fact_check": "",
        "report": "",
        "research_attempts": 0,
    },
    config={
        "configurable": {
            "thread_id": "cyberguard-demo-1"
        }
    }
)

print("=" * 60)
print("CYBERGUARD FINAL REPORT")
print("=" * 60)
print(final_result["report"])

print("\n" + "=" * 60)
print("RESEARCH ATTEMPTS:", final_result["research_attempts"])
print("=" * 60)


------------------------------------------------------------
=== TRACE: agent_run [agent] === (1878.87ms)
------------------------------------------------------------



OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791504000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3KE0wNvJLBuuR1FkYfZ8lzr3dZw'}

In [16]:
!pip install -q langchain-huggingface sentence-transformers

اضافه السياسات الداخليه

In [17]:
from pathlib import Path

POLICY_DIR = Path("securetech_policies")
POLICY_DIR.mkdir(exist_ok=True)

policies = {
    "phishing_policy.txt": """
SecureTech Organization
Phishing and Email Security Policy
Policy ID: SEC-POL-02

Employees must not click links, open attachments, or provide credentials
when an email appears suspicious or requests unusual actions.

Suspicious emails must be reported to the IT Security Team immediately.

Employees should not reply to suspicious messages or forward them to other
employees unless instructed by the security team.

Common phishing indicators include unexpected password reset requests,
urgent requests for money or credentials, suspicious sender addresses,
unexpected attachments, and links that lead to unfamiliar domains.

If an employee accidentally clicks a suspicious link or provides credentials,
they must immediately report the incident to the IT Security Team and change
the affected password when instructed.
""",

    "authentication_policy.txt": """
SecureTech Organization
Password and Authentication Policy
Policy ID: SEC-POL-01

Employees must use strong and unique passwords for organizational accounts.

Passwords must not be shared with other employees, written in publicly
accessible locations, or reused across unrelated personal services.

Multi-factor authentication (MFA) must be enabled for systems that support it,
especially systems containing sensitive organizational information.

Employees must immediately report suspected credential compromise to the
IT Security Team.

The IT Security Team may require a password reset when suspicious
authentication activity is detected.
""",

    "incident_response_policy.txt": """
SecureTech Organization
Incident Response Policy
Policy ID: SEC-POL-04

Security incidents must be reported to the IT Security Team as soon as they
are discovered.

Examples of security incidents include phishing attacks, suspected malware
infections, unauthorized access, credential theft, and data exposure.

Employees should preserve relevant evidence and should not intentionally
delete logs, emails, or files related to a suspected security incident.

The IT Security Team is responsible for investigating reported incidents,
determining their severity, containing the threat, and coordinating recovery
activities.

Employees must cooperate with the security team during incident investigations.
""",

    "access_control_policy.txt": """
SecureTech Organization
Access Control Policy
Policy ID: SEC-POL-03

Access to organizational systems must follow the principle of least privilege.

Employees should receive only the permissions required for their
responsibilities.

Access rights must be reviewed periodically and removed when they are
no longer required.

Employees must not use another employee's account or credentials.

Administrative privileges should be limited to authorized IT personnel and
should only be used when necessary.
"""
}

for filename, content in policies.items():
    (POLICY_DIR / filename).write_text(content.strip(), encoding="utf-8")

print(f"Created {len(policies)} policy files.")
print("Files:", [p.name for p in POLICY_DIR.iterdir()])

Created 4 policy files.
Files: ['incident_response_policy.txt', 'access_control_policy.txt', 'authentication_policy.txt', 'phishing_policy.txt']


In [18]:
!pip install -q langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [20]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    str(POLICY_DIR),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print("Loaded documents:", len(documents))

for doc in documents:
    print("\n---")
    print(doc.metadata["source"])
    print(doc.page_content[:150])

Loaded documents: 4

---
securetech_policies/incident_response_policy.txt
SecureTech Organization
Incident Response Policy
Policy ID: SEC-POL-04

Security incidents must be reported to the IT Security Team as soon as they
ar

---
securetech_policies/access_control_policy.txt
SecureTech Organization
Access Control Policy
Policy ID: SEC-POL-03

Access to organizational systems must follow the principle of least privilege.

E

---
securetech_policies/authentication_policy.txt
SecureTech Organization
Password and Authentication Policy
Policy ID: SEC-POL-01

Employees must use strong and unique passwords for organizational ac

---
securetech_policies/phishing_policy.txt
SecureTech Organization
Phishing and Email Security Policy
Policy ID: SEC-POL-02

Employees must not click links, open attachments, or provide credent


/tmp/ipykernel_4195/2650736417.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Total chunks:", len(chunks))

Original documents: 4
Total chunks: 5


In [22]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model loaded successfully.


In [23]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

vector_store.add_documents(chunks)

print(f"Added {len(chunks)} chunks to the vector store.")

Added 5 chunks to the vector store.


In [24]:
query = "What should an employee do after receiving a suspicious phishing email?"

retrieved_docs = vector_store.similarity_search(
    query,
    k=2
)

print("Retrieved documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n===== RESULT {i} =====")
    print("Source:", doc.metadata["source"])
    print(doc.page_content)

Retrieved documents:


===== RESULT 1 =====
Source: securetech_policies/phishing_policy.txt
SecureTech Organization
Phishing and Email Security Policy
Policy ID: SEC-POL-02

Employees must not click links, open attachments, or provide credentials
when an email appears suspicious or requests unusual actions.

Suspicious emails must be reported to the IT Security Team immediately.

Employees should not reply to suspicious messages or forward them to other
employees unless instructed by the security team.

Common phishing indicators include unexpected password reset requests,
urgent requests for money or credentials, suspicious sender addresses,
unexpected attachments, and links that lead to unfamiliar domains.

===== RESULT 2 =====
Source: securetech_policies/phishing_policy.txt
If an employee accidentally clicks a suspicious link or provides credentials,
they must immediately report the incident to the IT Security Team and change
the affected password when instructed.


In [25]:
def retrieve_policy_context(query: str, k: int = 3) -> str:
    retrieved_docs = vector_store.similarity_search(query, k=k)

    if not retrieved_docs:
        return "No relevant internal policy was found."

    context_parts = []

    for doc in retrieved_docs:
        source = Path(doc.metadata["source"]).name

        context_parts.append(
            f"[Internal Policy: {source}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(context_parts)

In [26]:
policy_context = retrieve_policy_context(
    "What should employees do if they receive a phishing email?"
)

print(policy_context)

[Internal Policy: phishing_policy.txt]
SecureTech Organization
Phishing and Email Security Policy
Policy ID: SEC-POL-02

Employees must not click links, open attachments, or provide credentials
when an email appears suspicious or requests unusual actions.

Suspicious emails must be reported to the IT Security Team immediately.

Employees should not reply to suspicious messages or forward them to other
employees unless instructed by the security team.

Common phishing indicators include unexpected password reset requests,
urgent requests for money or credentials, suspicious sender addresses,
unexpected attachments, and links that lead to unfamiliar domains.

[Internal Policy: phishing_policy.txt]
If an employee accidentally clicks a suspicious link or provides credentials,
they must immediately report the incident to the IT Security Team and change
the affected password when instructed.

[Internal Policy: incident_response_policy.txt]
SecureTech Organization
Incident Response Policy
Pol

In [27]:
class CyberGuardStateV3(TypedDict):
    query: str
    plan: str
    research: str
    policy_context: str
    analysis: str
    fact_check: str
    report: str
    research_attempts: int

In [28]:
def policy_rag_node(state: CyberGuardStateV3):
    query = state["query"]

    policy_context = retrieve_policy_context(
        query,
        k=3
    )

    return {
        "policy_context": policy_context
    }

print("Policy RAG node created successfully.")

Policy RAG node created successfully.


In [29]:
ANALYST_PROMPT_V2 = """
You are the CyberGuard Analyst.

You analyze cybersecurity research using two evidence sources:

1. External Web Research
2. SecureTech Internal Policy Evidence

Your job is to:
- Identify important facts from the external research.
- Identify relevant requirements from the internal policies.
- Compare external security guidance with internal policies.
- Identify security risks, gaps, contradictions, or missing controls.
- Do not invent facts, policies, or sources.
- Clearly distinguish internal policy requirements from external recommendations.

Produce a concise structured analysis that can be used by a report writer.
"""

analyst_v2 = create_agent(
    model=llm,
    tools=[],
    system_prompt=ANALYST_PROMPT_V2
)

print("CyberGuard Analyst with RAG support created successfully.")

CyberGuard Analyst with RAG support created successfully.


In [30]:
async def analyst_node_v4(state: CyberGuardStateV3) -> dict:

    analysis_query = f"""
Original question:
{state["query"]}

External Web Research:
{state["research"]}

SecureTech Internal Policy Evidence:
{state["policy_context"]}

Analyze the question using both sources.

Compare the external research with the internal policies.
Identify important facts, security risks, gaps, contradictions,
and recommendations.

Clearly distinguish between:
- External security guidance
- SecureTech internal policy requirements

Do not invent information.
"""

    result = await run_agent(
        analyst_v2,
        analysis_query,
        10
    )

    return {"analysis": result["answer"]}

print("RAG-enabled Analyst node created successfully.")

RAG-enabled Analyst node created successfully.


In [31]:
FACT_CHECKER_PROMPT_V2 = """
You are the CyberGuard Fact Checker.

Your job is to evaluate cybersecurity findings using two evidence sources:

1. External Web Research
2. SecureTech Internal Policy Evidence

Check:
- Are important external claims supported by the research?
- Are internal policy requirements accurately represented?
- Are there contradictions between external sources and internal policies?
- Is important information missing?
- Are the sources relevant and trustworthy?

If the evidence is sufficient, return exactly:
PASS

If important evidence is missing or more research is needed, return exactly:
RESEARCH_AGAIN

Then briefly explain your decision.

Do not invent facts, policies, or sources.
"""

fact_checker_v2 = create_agent(
    model=llm,
    tools=[],
    system_prompt=FACT_CHECKER_PROMPT_V2
)

print("RAG-enabled Fact Checker created successfully.")

RAG-enabled Fact Checker created successfully.


In [32]:
async def fact_checker_node_v2(state: CyberGuardStateV3) -> dict:

    fact_check_query = f"""
Original question:
{state["query"]}

External Web Research:
{state["research"]}

SecureTech Internal Policy Evidence:
{state["policy_context"]}

Analysis:
{state["analysis"]}

Evaluate whether the findings are sufficiently supported.

Check both:
1. External research evidence
2. Internal policy evidence

Look for unsupported claims, contradictions, missing information,
and unreliable sources.

At the end, return either:
PASS
or
RESEARCH_AGAIN

Then briefly explain why.
"""

    result = await run_agent(
        fact_checker_v2,
        fact_check_query,
        5
    )

    return {
        "fact_check": result["answer"]
    }

print("RAG-enabled Fact Checker node created successfully.")

RAG-enabled Fact Checker node created successfully.


In [33]:
WRITER_PROMPT_V2 = """
You are the CyberGuard Report Writer.

Your job is to produce a professional cybersecurity report based on:

1. External Web Research
2. SecureTech Internal Policy Evidence
3. CyberGuard Analysis
4. Fact-Check Results

The report should:
- Answer the user's original cybersecurity question.
- Clearly distinguish between external security guidance and internal SecureTech policy.
- Identify important security risks and gaps.
- Provide practical recommendations.
- Mention relevant internal policy IDs when used.
- Preserve external source links when available.
- Never invent facts, policies, or sources.
- If internal policy and external guidance differ, clearly identify the difference.
- Use simple professional language.
- Use headings and bullet points where useful.

Do not write as a conversational chatbot.
Write as a concise cybersecurity assessment report.
"""

writer_v2 = create_agent(
    model=llm,
    tools=[],
    system_prompt=WRITER_PROMPT_V2
)

print("RAG-enabled Report Writer created successfully.")

RAG-enabled Report Writer created successfully.


In [34]:
async def writer_node_v4(state: CyberGuardStateV3) -> dict:

    writer_query = f"""
Original question:
{state["query"]}

External Web Research:
{state["research"]}

SecureTech Internal Policy Evidence:
{state["policy_context"]}

CyberGuard Analysis:
{state["analysis"]}

Fact-Check Result:
{state["fact_check"]}

Write the final CyberGuard cybersecurity assessment report.

The report must:
- Answer the original question.
- Use the available evidence only.
- Distinguish internal policy requirements from external guidance.
- Identify important risks and gaps.
- Give practical recommendations.
- Include relevant policy IDs and source links when available.
"""

    result = await run_agent(
        writer_v2,
        writer_query,
        10
    )

    return {
        "report": result["answer"]
    }

print("RAG-enabled Writer node created successfully.")

RAG-enabled Writer node created successfully.


In [35]:
from langchain_core.tools import tool

# Reset the detector before creating a new one
loop_detector = LoopDetector(
    exact_threshold=1,
    fuzzy_threshold=0.8,
    stagnation_window=3
)

@tool("guarded_search_web")
def guarded_search_web(query: str) -> str:
    """
    Search the web using Tavily with repetition detection.
    """

    loop_check = loop_detector.check_tool_call(
        tool_name="search_web",
        tool_input=query
    )

    if loop_check.is_looping:
        return (
            f"LOOP DETECTED: {loop_check.message}\n"
            "Do not repeat this search. Try a different search query."
        )

    return search_web.invoke({
        "query": query
    })

print("Guarded Tavily search tool created successfully.")

Guarded Tavily search tool created successfully.


In [36]:
print("LoopDetector tool guard created successfully.")

LoopDetector tool guard created successfully.


In [37]:
RESEARCHER_PROMPT_V2 = """
You are the CyberGuard Researcher.

Investigate cybersecurity questions using web research.

Rules:
- Use reliable cybersecurity sources.
- Prefer CISA, NIST, government sources, security organizations,
  vendor documentation, and reputable research.
- Do not repeat the same search query.
- If a search is blocked by the LoopDetector, change the search query.
- Use read_webpage when an important source needs deeper inspection.
- Do not invent facts or sources.
- Clearly separate verified information from uncertain information.
- Include source titles and URLs when available.
- Keep the research concise and useful for the Analyst.
"""

researcher_v2 = create_agent(
    model=llm,
    tools=[guarded_search_web, read_webpage],
    system_prompt=RESEARCHER_PROMPT_V2,
    middleware=[
        ToolCallLimitMiddleware(
            thread_limit=10,
            exit_behavior="end"
        )
    ],
)

print("Loop-protected Researcher created successfully.")

Loop-protected Researcher created successfully.


In [38]:
async def loop_guard_node(state):
    research = state.get("research", "")

    if not research:
        return {
            "loop_warning": ""
        }

    loop_check = loop_detector.check_output_stagnation(research)

    if loop_check.is_looping:
        warning = (
            f"STAGNATION DETECTED: {loop_check.message}\n"
            "The research output appears repetitive. "
            "Another research attempt is required."
        )
        print(warning)

        return {
            "loop_warning": warning
        }

    return {
        "loop_warning": ""
    }


print("Loop guard node created successfully.")

Loop guard node created successfully.


In [39]:
class CyberGuardStateFinal(TypedDict):
    query: str
    plan: str
    research: str
    policy_context: str
    analysis: str
    fact_check: str
    report: str
    research_attempts: int
    loop_warning: str

print("Final CyberGuard state created successfully.")

Final CyberGuard state created successfully.


In [40]:
async def researcher_node_final(state: CyberGuardStateFinal) -> dict:

    research_query = f"""
Original question:
{state["query"]}

Research plan:
{state["plan"]}

Previous research attempt:
{state.get("research", "")}

Loop warning:
{state.get("loop_warning", "")}

Conduct or improve the research based on the plan.

If a loop warning exists:
- Do not repeat the previous research.
- Use different search queries.
- Look for additional reliable sources.

Focus on reliable cybersecurity sources and provide source links.
"""

    result = await run_agent(
        researcher_v2,
        research_query,
        10
    )

    return {
        "research": result["answer"],
        "research_attempts": state.get("research_attempts", 0) + 1
    }


print("Final Researcher node created successfully.")

Final Researcher node created successfully.


In [41]:
def final_fact_check_router(state: CyberGuardStateFinal):

    decision = state["fact_check"].strip().upper()
    attempts = state.get("research_attempts", 0)

    first_line = decision.splitlines()[0] if decision else ""

    if first_line == "PASS":
        return "writer"

    if attempts < 2:
        return "researcher"

    print("Maximum research attempts reached. Continuing to Writer.")

    return "writer"

In [51]:
async def planner_node_v2(state: CyberGuardStateFinal) -> dict:
    result = await run_agent(planner, state["query"], 5)
    return {"plan": result["answer"]}

In [52]:
graph_final = StateGraph(CyberGuardStateFinal)

# Nodes
graph_final.add_node("planner", planner_node_v2)
graph_final.add_node("researcher", researcher_node_final)
graph_final.add_node("policy_rag", policy_rag_node)
graph_final.add_node("loop_guard", loop_guard_node)
graph_final.add_node("analyst", analyst_node_v4)
graph_final.add_node("fact_checker", fact_checker_node_v2)
graph_final.add_node("writer", writer_node_v4)

# Start
graph_final.add_edge(START, "planner")

# Planner → parallel research + RAG
graph_final.add_edge("planner", "researcher")
graph_final.add_edge("planner", "policy_rag")

# Research → LoopDetector
graph_final.add_edge("researcher", "loop_guard")

# Both evidence paths → Analyst
graph_final.add_edge("loop_guard", "analyst")
graph_final.add_edge("policy_rag", "analyst")

# Analyst → Fact Checker
graph_final.add_edge("analyst", "fact_checker")

# Fact Checker → retry or writer
graph_final.add_conditional_edges(
    "fact_checker",
    final_fact_check_router,
    {
        "researcher": "researcher",
        "writer": "writer"
    }
)

# Writer → End
graph_final.add_edge("writer", END)

cyberguard_graph_final = graph_final.compile()

print("CyberGuard final graph compiled successfully.")

CyberGuard final graph compiled successfully.


In [59]:
cyberguard_graph_final = graph_final.compile(
    checkpointer=checkpointer
)

print("CyberGuard final graph with checkpointer compiled successfully.")

CyberGuard final graph with checkpointer compiled successfully.


In [53]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

print("Checkpointer created successfully.")

Checkpointer created successfully.


In [58]:
cyberguard_graph_final = graph_final.compile(
    checkpointer=checkpointer
)

print("CyberGuard final graph with checkpointer compiled successfully.")

CyberGuard final graph with checkpointer compiled successfully.


In [60]:
test_query = """
An employee at SecureTech receives a suspicious email asking them to
click a link and provide their password.

What should the employee do according to SecureTech's internal policies,
and what additional cybersecurity protections are recommended by external
security guidance?
"""

initial_state = {
    "query": test_query,
    "plan": "",
    "research": "",
    "policy_context": "",
    "analysis": "",
    "fact_check": "",
    "report": "",
    "research_attempts": 0,
    "loop_warning": ""
}

config = {
    "configurable": {
        "thread_id": "cyberguard-demo-01"
    }
}

print("Running CyberGuard...")

Running CyberGuard...


In [61]:
result = await cyberguard_graph_final.ainvoke(
    initial_state,
    config=config
)

print("CyberGuard finished successfully!")
print("\n" + "=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(result["report"])


------------------------------------------------------------
=== TRACE: agent_run [agent] === (1768.32ms)
------------------------------------------------------------



OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791504000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3KE0wNvJLBuuR1FkYfZ8lzr3dZw'}

In [63]:
import os
from google.colab import userdata

TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

print("TAVILY KEY FOUND:", bool(TAVILY_API_KEY))
print("KEY LENGTH:", len(TAVILY_API_KEY) if TAVILY_API_KEY else 0)

TAVILY KEY FOUND: True
KEY LENGTH: 57


In [64]:
test_research = await run_agent(
    researcher_v2,
    "How can an organization protect itself from phishing attacks?",
    10
)

print("=" * 60)
print("RESEARCHER RESULT")
print("=" * 60)
print(test_research["answer"])


------------------------------------------------------------
=== TRACE: agent_run [agent] === (1406.93ms)
------------------------------------------------------------



OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791504000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3KE0wNvJLBuuR1FkYfZ8lzr3dZw'}

In [65]:
import inspect

print(inspect.signature(loop_detector.check_tool_call))

(tool_name: str, tool_input: str) -> __main__.LoopDetectionResult


In [66]:
test_guarded = guarded_search_web.invoke({
    "query": "CISA phishing protection best practices"
})

print(test_guarded)

1. Phishing Guidance: Stopping the Attack Cycle at Phase One
   Link: https://www.cisa.gov/sites/default/files/2025-03/Phishing%20Guidance%20-%20Stopping%20the%20Attack%20Cycle%20at%20Phase%20One%20508.pdf
   Snippet: SMALL- AND MEDIUM-SIZED BUSINESSES (SMBs) OR ORGANIZATIONS CISA, NSA, FBI, and MS-ISAC recommend that small- and medium-sized organizations with limited resources prioritize the following best practices to protect network resources from prevalent phishing threats: • User phishing awareness training: Implement a standard anti-phishing training program and require employees to review phishing training material annually. Additionally, conclude the CISA | NSA | FBI | MS-ISAC TLP:CLEAR 8 PHISHING [...] Additionally, CISA, NSA, FBI, and MS-ISAC recommend SMBs implement the technical solutions below to prevent phishing related compromises: • Implement strong password policies to authenticate users. These passwords must adhere to a password strength policy which requires minimum 

In [67]:
test_guarded_2 = guarded_search_web.invoke({
    "query": "CISA phishing protection best practices"
})

print(test_guarded_2)

LOOP DETECTED: Exact loop detected: 'search_web' called 2 times with identical arguments. Change your approach.
Do not repeat this search. Try a different search query.


In [68]:
loop_detector.reset()

test_guarded_1 = guarded_search_web.invoke({
    "query": "CISA phishing protection best practices"
})

print("FIRST CALL:")
print(test_guarded_1[:500])

test_guarded_2 = guarded_search_web.invoke({
    "query": "CISA phishing protection best practices"
})

print("\nSECOND CALL:")
print(test_guarded_2[:500])

FIRST CALL:
1. Phishing Guidance: Stopping the Attack Cycle at Phase One
   Link: https://covacci.org/wp-content/uploads/2023/10/CISA_Phishing-Guidance-Stopping-the-attach-cycle-at-phase-one.pdf
   Snippet: CISA, NSA, FBI, and MS-ISAC recommend that small- and medium-sized organizations with limited resources prioritize the following best practices to protect network resources from prevalent phishing threats:  User phishing awareness training: Implement a standard anti-phishing training program and require

SECOND CALL:
LOOP DETECTED: Exact loop detected: 'search_web' called 2 times with identical arguments. Change your approach.
Do not repeat this search. Try a different search query.
